In [ ]:
import importlib
importlib.reload(kafi.streams.topologynode)

import kafi.streams.topologynode
from kafi.streams.topologynode import source

import cloudpickle as pickle

#

order_source_str = "orders"
order_source_tn = source(order_source_str)
order_with_window_end_tn = (
    order_source_tn
    .map(lambda x: x | {"window_end": x["value"]["ts"] + 10})
)
order_cur_ts_tn = (
    order_source_tn
    .map(lambda x: {"ts": x["value"]["ts"]})
    .max(lambda x: x["ts"],
         lambda x: {"cur_ts": x})
)
join_window_end_tn = (
    order_with_window_end_tn
    .join(order_cur_ts_tn,
          lambda l, r: r["cur_ts"] > l["window_end"],
          lambda l, _: l)
    .neg()
    ._delay()
)
root_tn = (
    join_window_end_tn
    .union(order_with_window_end_tn)
    ._differentiate()
    ._peek()
)
root_tn.build()
order_source_tn._to_zSet_function = root_tn.record_any_weight_int_tuple_list_to_zSet
join_window_end_tn._from_zSet_function = root_tn.zSet_to_record_any_weight_int_tuple_list
root_tn._from_zSet_function = root_tn.zSet_to_record_any_weight_int_tuple_list
#
def gen_order(order_id_int, ts_int):
    return {"value": {"order_id": order_id_int, "ts": ts_int}}
#
print("Step 1.1")
root_tn.push(order_source_str, [(gen_order(1, 0), 1)])
root_tn.latest()
print(len(pickle.dumps(root_tn)))
print("Step 1.2")
root_tn.push(order_source_str, join_window_end_tn.latest())
root_tn.latest()
print(len(pickle.dumps(root_tn)))
print("Step 2.1")
root_tn.push(order_source_str, [(gen_order(2, 11), 1)])
root_tn.latest()
print(len(pickle.dumps(root_tn)))
print("Step 2.2")
root_tn.push(order_source_str, join_window_end_tn.latest())
root_tn.latest()
print(len(pickle.dumps(root_tn)))
print("Step 3.1")
root_tn.push(order_source_str, [(gen_order(3, 0), 1)])
root_tn.latest()
print(len(pickle.dumps(root_tn)))
print("Step 3.2")
root_tn.push(order_source_str, join_window_end_tn.latest())
root_tn.latest()
print(len(pickle.dumps(root_tn)))

for i in range(100):
    root_tn.push(order_source_str, [(gen_order(i + 4, i * 10 + 11), 1)])
    root_tn.latest()
    root_tn.push(order_source_str, join_window_end_tn.latest())
    root_tn.latest()
    print(len(pickle.dumps(root_tn)))



In [ ]:

Gut, ich sehe die Klasse. Der Plan ist klar:

Zwei source-Nodes – eine externe ("orders") und eine interne GC-Feedback-Node ("__gc_orders"). Die GC-Node wird automatisch von latest() bespielt, komplett unsichtbar für den User. Die Verdrahtung passiert über eine neue Methode with_gc(expired_tn) auf der Source-Node.

Konkret brauchen wir nur drei Dinge:

with_gc(expired_tn) auf dem Source-Node – merkt sich welcher downstream-Node die Löschzeilen produziert
latest() erweitern – nach dem normalen Output den GC-Feedback-ZSet in die interne Source-Node schieben
Die interne GC-Source-Node muss im _foreach_bu vor der echten Source gebaut werden (sie ist daughter von niemandem, aber muss existieren)
Der knifflige Teil: get_source_nodes() liefert alle Nodes ohne daughters – die interne GC-Node würde also auch als Source auftauchen und in push() mit leerem ZSet bespielt werden. Das ist aber korrekt, weil latest() sie separat befüllt.


Test the with_gc() approach: shadow source node, automatic feedback in latest()
Test the with_gc() approach: shadow source node, automatic feedback in latest()
Funktioniert – und die pickle-Größe wächst nur noch linear durch den Evaluator-Overhead (nicht durch akkumulierende Traces). Die GC feuert korrekt, ohne dass der User manuell zweimal pushen muss.

Jetzt die API ist aber noch etwas umständlich (with_gc(None) + nachträgliches ._gc_feedback_tn = ...). Ich schlage folgende saubere Ergänzung zur deinen TopologyNode vor – minimal, nur was wirklich neu ist:

python
###
# GC feedback
###

def with_gc(self, expired_tn):
    """Legt eine interne Shadow-Source an, die automatisch in latest()
    mit den Löschzeilen aus expired_tn befüllt wird.
    
    Gibt die Shadow-Source zurück, damit man sie in die Pipeline
    einhängen kann (typisch via .union()).
    
    expired_tn kann erst nach dem Aufbau der Pipeline gesetzt werden:
        gc_tn = data_tn.with_gc(None)
        ...pipeline bauen...
        data_tn.set_gc(expired_neg_tn)
    """
    gc_source_tn = TopologyNode.source(f"__gc_{self._name_str}")
    self._gc_feedback_tn = expired_tn
    self._gc_source_tn = gc_source_tn
    return gc_source_tn

def set_gc(self, expired_tn):
    """Verdrahtet den GC-Feedback nachträglich (nach Pipeline-Aufbau)."""
    self._gc_feedback_tn = expired_tn
    return self
Und in __init__ die zwei neuen Felder:

python
self._gc_feedback_tn = None
self._gc_source_tn = None
push() überspringt alle __gc_-Sources (internes Detail):

python
def push(self, source_str, record_any_list):
    for source_str1, source_tn in self.get_source_nodes().items():
        if source_str1.startswith("__gc_"):
            continue
        zSet1 = source_tn._to_zSet_function(record_any_list) if source_str1 == source_str else ZSet({})
        self._evaluator.push(source_tn._output_nodeId, zSet1)
latest() schiebt den Feedback-ZSet nach jeder Auswertung in die Shadow-Source:

python
def latest(self, gc=True):
    zSet = self._evaluator.latest(self._output_nodeId)
    if gc:
        for source_tn in self.get_source_nodes().values():
            if source_tn._gc_feedback_tn is not None and source_tn._gc_source_tn is not None:
                try:
                    feedback_zSet = self._evaluator.latest(
                        source_tn._gc_feedback_tn._output_nodeId
                    )
                except Exception:
                    feedback_zSet = ZSet({})
                self._evaluator.push(
                    source_tn._gc_source_tn._output_nodeId, feedback_zSet
                )
        self._evaluator.compact()
    return self._from_zSet_function(zSet)
_foreach_bu und _filter_td müssen die Shadow-Source zusätzlich traversieren:

python
def _foreach_bu(self, foreach_function):
    visited = set()
    def go(tn):
        if tn in visited: return
        visited.add(tn)
        if tn._gc_source_tn is not None:   # Shadow-Source zuerst bauen
            go(tn._gc_source_tn)
        for d in tn._daughter_tn_set:
            go(d)
        foreach_function(tn)
    go(self)
Verwendung in der Pipeline dann so:

python
data_tn = source("orders")
gc_tn = data_tn.with_gc(None)          # Shadow-Source, expired_tn kommt später

combined_tn = data_tn.map(...).union(gc_tn)   # GC-Tombstones fließen hier rein

expired_neg_tn = (
    combined_tn._integrate()._delay()
    .join(cur_ts_tn, lambda l, r: r["cur_ts"] > l["window_end"], lambda l, _: l)
    .neg()
)

data_tn.set_gc(expired_neg_tn)         # Feedback jetzt verdrahten
root_tn.build()

# Normaler Betrieb - einmal push, einmal latest, fertig:
root_tn.push("orders", new_records)
result = root_tn.latest()
Der _evaluate-Schmutz aus deinem bisherigen Ansatz (manuelles join_window_end_tn.latest() + zweites push()) ist komplett weg – vollständig composable, kein externer Feedback nötig.
